# Feature Engineering (XGBoost)

Este notebook cubre únicamente la preparación de datos para el modelo XGBoost: se cargan y separan los datos con el pipeline compartido, se quitan columnas que no sirven y se deja listo el train real. XGBoost entrena **sin SMOTE** (decisión de `02b_ablacion_smote.ipynb`). Con el dataset original, la regla de `02b` eligió la variante B (sin pesos de clase); con el dataset corregido (443,614 filas) eligió la C, con `scale_pos_weight`, que se calcula en los notebooks de entrenamiento a partir de `y_train`. La versión con SMOTE se genera y se guarda aparte, solo para esa ablación.

`cargar_dataset_limpio`, `train_test_split_by_flow` y `reportar_duplicados_de_features` vienen de `src/preprocessing/clean.py`, que también usan GRU y SVM. No se modifican acá. Si se tocara ese archivo, la comparación final entre los tres modelos dejaría de ser válida.

In [1]:
import sys
sys.path.append("../..")  # raíz del repo, desde notebooks/xgboost/

from src.preprocessing.clean import (
    cargar_dataset_limpio,
    reportar_duplicados_de_features,
    train_test_split_by_flow,
)

# En Google Colab, en vez de la celda anterior:
# from google.colab import drive
# drive.mount('/content/drive')
# !git clone https://github.com/<usuario-u-org>/<nombre-repo>.git
# %cd <nombre-repo>
# !pip install -r requirements.txt -q
# from src.preprocessing.clean import (
#     cargar_dataset_limpio,
#     reportar_duplicados_de_features,
#     train_test_split_by_flow,
# )

### Carga y split cronológico (pipeline compartido)

`cargar_dataset_limpio` lee el CSV que genera `notebooks/Cleaning/limpieza_low_and_slow.ipynb` (carga, filtro de BENIGN/Slowloris/Slowhttptest, limpieza y codificación de la etiqueta ya aplicados). `train_test_split_by_flow` corta por tiempo, no al azar; el porqué está en `docs/adr/0001-*.md`.

`X` sale con 61 columnas: las 59 features más `Timestamp` y `bloque`, que son metadata del split y no features reales.

In [2]:
df = cargar_dataset_limpio("../../data/processed/wednesday_low_and_slow_clean.csv")

N_FILAS_ESPERADAS = 443_614
N_FEATURES_ESPERADAS = 59
features_df = [c for c in df.columns if c not in ("Timestamp", "Label", "label_binaria")]

assert len(df) == N_FILAS_ESPERADAS, (
    f"Se esperaban {N_FILAS_ESPERADAS:,} filas, se obtuvieron {len(df):,}. "
    "Regenerar el CSV con notebooks/Cleaning/limpieza_low_and_slow.ipynb."
)
assert len(features_df) == N_FEATURES_ESPERADAS, (
    f"Se esperaban {N_FEATURES_ESPERADAS} features, se obtuvieron {len(features_df)}. "
    "Regenerar el CSV con notebooks/Cleaning/limpieza_low_and_slow.ipynb."
)
print(f"Dataset cargado: {len(df):,} filas, {len(features_df)} features.\n")

X_train, X_val, X_test, y_train, y_val, y_test = train_test_split_by_flow(df)

shape_antes_poda = {
    "X_train": X_train.shape,
    "X_val": X_val.shape,
    "X_test": X_test.shape,
}
print("\nShapes después del split (antes de podar columnas):")
for nombre, shape in shape_antes_poda.items():
    print(f"  {nombre}: {shape}")

assert X_train.shape[1] == 61, (
    f"Se esperaban 61 columnas en X_train (59 features + Timestamp + bloque), "
    f"se obtuvieron {X_train.shape[1]}. Revisar si clean.py cambió o si hay que "
    "regenerar el CSV con el notebook de limpieza."
)
print("\nOK: X_train tiene 61 columnas, como se esperaba.")

print("\nProporción de ataques por split:")
for nombre, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  {nombre}: {y.mean():.4%} ({int(y.sum())} de {len(y)})")

Dataset cargado: 443,614 filas, 59 features.



Fronteras de bloque (outlier_quantile=0.0, desempate=csv):
  Slowloris:    [2017-07-05 09:48:00 .. 2017-07-05 10:11:00]  (5790 filas)
  Slowhttptest: [2017-07-05 10:15:00 .. 2017-07-05 10:37:00]  (5499 filas)

Composición por bloque:
  Bloque 1: 53375 filas | Timestamp [2017-07-05 08:42:00 .. 2017-07-05 09:47:00] | ataques: 0 (0.00%)
  Bloque 2: 30362 filas | Timestamp [2017-07-05 09:48:00 .. 2017-07-05 10:11:00] | ataques: 5790 (19.07%)
  Bloque 3: 3578 filas | Timestamp [2017-07-05 10:12:00 .. 2017-07-05 10:14:00] | ataques: 0 (0.00%)
  Bloque 4: 33203 filas | Timestamp [2017-07-05 10:15:00 .. 2017-07-05 10:37:00] | ataques: 5499 (16.56%)
  Bloque 5: 323096 filas | Timestamp [2017-07-05 10:38:00 .. 2017-07-05 17:10:00] | ataques: 0 (0.00%)



Train: 310529 filas | Val: 66541 filas | Test: 66544 filas
Proporción de ataques -> train: 0.0265, val: 0.0283, test: 0.0177

Shapes después del split (antes de podar columnas):
  X_train: (310529, 61)
  X_val: (66541, 61)
  X_test: (66544, 61)

OK: X_train tiene 61 columnas, como se esperaba.

Proporción de ataques por split:
  train: 2.6503% (8230 de 310529)
  val: 2.8313% (1884 de 66541)
  test: 1.7657% (1175 de 66544)


### Poda de columnas adicionales para XGBoost

Aparte de Timestamp y bloque (metadata del split, no features reales — ya se explicó arriba), hay 4 columnas más que se eliminan por otra razón y es que son casi la misma pregunta hecha dos veces. CICFlowMeter mide el mismo comportamiento del tráfico desde ángulos casi idénticos (máximo vs. desviación estándar, ida vs. vuelta), y si se dejan las dos, XGBoost reparte el crédito de "esto ayudó a detectar el ataque" entre columnas gemelas en vez de concentrarlo en una, lo cual ensucia el análisis de importancia de características que pide el protocolo.

El criterio para elegir cuál se queda de cada par es la que da la señal más directa del comportamiento del atacante, no la que reacciona indirectamente a él.

| Se va | Se queda | Por qué |
|---|---|---|
| `Flow IAT Std` | `Flow IAT Max` | El máximo captura la pausa más larga del flujo — la señal más literal de Slowloris |
| `Bwd IAT Std` | `Fwd IAT Std` | El atacante controla el ritmo de ida (fwd); la vuelta (bwd) solo reacciona |
| `Bwd IAT Mean` | `Fwd IAT Mean` | Mismo argumento que la fila anterior, con el promedio en vez de la variabilidad |
| `Packet Length Std` | `Packet Length Mean` | Los paquetes de Slowloris son chicos y parejos — el promedio ya refleja eso |

#### Puertos y `Protocol`

`Destination Port`, `Source Port`, `Protocol` y `Fwd PSH Flags` ya no llegan a este notebook: el pipeline compartido (`clean.py`) los quita porque son atajos que salen de cómo se armó el laboratorio de CIC-IDS2017 y no del comportamiento del ataque. La ablación y el análisis de errores están en `docs/auditoria-baseline-xgboost.md`.

En total se eliminan 6 columnas: `Timestamp`, `bloque` y las 4 de la tabla anterior. Se podan acá, antes de generar la versión con SMOTE, porque SMOTE calcula los vecinos con todas las columnas. `X` queda con 55 columnas.

In [3]:
columnas_a_eliminar = [
    "Timestamp",
    "bloque",
    "Flow IAT Std",
    "Bwd IAT Std",
    "Bwd IAT Mean",
    "Packet Length Std",
]

X_train = X_train.drop(columns=columnas_a_eliminar)
X_val = X_val.drop(columns=columnas_a_eliminar)
X_test = X_test.drop(columns=columnas_a_eliminar)

shape_despues_poda = {
    "X_train": X_train.shape,
    "X_val": X_val.shape,
    "X_test": X_test.shape,
}
print("Shapes después de podar columnas:")
for nombre, shape in shape_despues_poda.items():
    print(f"  {nombre}: {shape}")

Shapes después de podar columnas:
  X_train: (310529, 55)
  X_val: (66541, 55)
  X_test: (66544, 55)


In [4]:
COLUMNAS_PROHIBIDAS = {
    "Destination Port", "Source Port", "Protocol", "Fwd PSH Flags", "Timestamp", "bloque",
}

for nombre, X in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test)]:
    assert X.shape[1] == 55, (
        f"Se esperaban 55 columnas en {nombre} tras la poda, se obtuvieron "
        f"{X.shape[1]}. Shape real: {X.shape}. Detenerse y revisar antes de "
        "seguir."
    )
    assert list(X.columns) == list(X_train.columns), f"{nombre} tiene otras columnas u otro orden que X_train"
    presentes = COLUMNAS_PROHIBIDAS & set(X.columns)
    assert not presentes, f"{nombre} todavía tiene columnas prohibidas: {sorted(presentes)}"
print("OK: X_train, X_val y X_test quedaron con las mismas 55 columnas, en el mismo orden y sin columnas prohibidas.")

OK: X_train, X_val y X_test quedaron con las mismas 55 columnas, en el mismo orden y sin columnas prohibidas.


### Diagnóstico de duplicados de features

Se reporta cuántos vectores de features se repiten dentro de cada split y cuántos de val/test ya aparecieron en train. No se elimina ninguna fila: es solo un diagnóstico para saber qué tan optimistas pueden ser las métricas de evaluación.

In [5]:
reporte_duplicados = reportar_duplicados_de_features(
    X_train, X_val, X_test, y_train, y_val, y_test
)

split  clase  filas  dup_en_split  en_train
train      0 302299         29556       NaN
train      1   8230           288       NaN
  val      0  64657          4807    7848.0
  val      1   1884           180     121.0
 test      0  65369          4142    8084.0
 test      1   1175           109      33.0

Filas cuyo vector aparece con ambas etiquetas: 0


### Versión con SMOTE (solo para la ablación de `02b`)

XGBoost entrena con el train real, sin SMOTE. La regla de `02b_ablacion_smote.ipynb` eligió entrenar sin pesos de clase (variante B) con el dataset original, y con `scale_pos_weight` (variante C) con el dataset corregido de 443,614 filas. El peso no se aplica acá: lo calculan los notebooks de entrenamiento a partir de `y_train`. `train` queda mayoritariamente benigno porque el split es cronológico, no aleatorio (la proporción exacta se imprime arriba).

Aun así se genera la versión balanceada con SMOTE y se guarda aparte (`X_train_smote.csv` / `y_train_smote.csv`), solo para que `02b` pueda repetir la comparación. `sampling_strategy=0.4` es la razón ataque/benigno que se busca, no el porcentaje final de ataques; `k_neighbors=5` y `random_state=42`. SMOTE agrega las filas sintéticas al final, así que las primeras filas de la versión con SMOTE deben ser idénticas al train real; se verifica con un `assert`.

Importante: `val` y `test` no se tocan. Si se les aplicara SMOTE, se estarían metiendo datos inventados justo en lo que se usa para medir qué tan bien funciona el modelo, y esas métricas dejarían de significar algo real. Se verifica con un `assert` abajo, no solo se afirma.

In [6]:
import pandas as pd
from imblearn.over_sampling import SMOTE

val_shape_antes_smote = X_val.shape
test_shape_antes_smote = X_test.shape
n_train_real = len(X_train)

prop_ataques_real = y_train.mean()
n_ataques_real = int(y_train.sum())
print(f"Train real (sin SMOTE) -> X_train: {X_train.shape} | ataques: {n_ataques_real}/"
      f"{n_train_real} ({prop_ataques_real:.4%})")

# X_train / y_train NO se reemplazan: son el train real con el que entrena XGBoost.
smote = SMOTE(sampling_strategy=0.4, k_neighbors=5, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

prop_ataques_smote = y_train_smote.mean()
n_ataques_smote = int(y_train_smote.sum())
n_sinteticos = len(X_train_smote) - n_train_real
print(f"Con SMOTE (solo para 02b) -> X_train_smote: {X_train_smote.shape} | ataques: {n_ataques_smote}/"
      f"{len(y_train_smote)} ({prop_ataques_smote:.4%})")
print(f"Ejemplos sintéticos generados: {n_sinteticos}")

# SMOTE agrega las sintéticas al final: las primeras filas deben ser el train real, sin cambios.
assert len(X_train_smote) > n_train_real, "SMOTE no generó filas sintéticas"
pd.testing.assert_frame_equal(
    X_train_smote.iloc[:n_train_real].reset_index(drop=True),
    X_train.reset_index(drop=True),
    check_dtype=False,
)
assert (y_train_smote.iloc[:n_train_real].to_numpy() == y_train.to_numpy()).all(), (
    "Las primeras filas de y_train_smote no coinciden con y_train real"
)
assert (y_train_smote.iloc[n_train_real:] == 1).all(), "Las filas sintéticas deberían ser todas ataque"
print(f"OK: las primeras {n_train_real:,} filas de X_train_smote son idénticas a X_train real.")

# Verificación explícita: val/test NUNCA deben cambiar con SMOTE
assert X_val.shape == val_shape_antes_smote, (
    f"X_val cambió de shape ({val_shape_antes_smote} -> {X_val.shape}) — "
    "SMOTE no debe tocar val bajo ninguna circunstancia."
)
assert X_test.shape == test_shape_antes_smote, (
    f"X_test cambió de shape ({test_shape_antes_smote} -> {X_test.shape}) — "
    "SMOTE no debe tocar test bajo ninguna circunstancia."
)
print(f"\nOK: X_val {X_val.shape} y X_test {X_test.shape} no fueron modificados por SMOTE.")

Train real (sin SMOTE) -> X_train: (310529, 55) | ataques: 8230/310529 (2.6503%)


Con SMOTE (solo para 02b) -> X_train_smote: (423218, 55) | ataques: 120919/423218 (28.5713%)
Ejemplos sintéticos generados: 112689
OK: las primeras 310,529 filas de X_train_smote son idénticas a X_train real.

OK: X_val (66541, 55) y X_test (66544, 55) no fueron modificados por SMOTE.


### Escalado de features (omitido para XGBoost)

`scale_features()` existe en `clean.py`, pero no se usa acá. XGBoost divide por umbrales ("¿esta columna es mayor a X?"), no por distancia y escalar no le cambia nada al resultado. SVM y GRU sí la necesitan.

### Resumen

In [7]:
n_cols_eliminadas = shape_antes_poda["X_train"][1] - shape_despues_poda["X_train"][1]

print("=" * 60)
print("RESUMEN — Fase 5: Feature Engineering (XGBoost)")
print("=" * 60)

print("\nShapes antes de podar columnas:")
for nombre, shape in shape_antes_poda.items():
    print(f"  {nombre}: {shape}")

print(f"\nShapes después de podar columnas (-{n_cols_eliminadas} columnas: "
      f"{', '.join(columnas_a_eliminar)}):")
for nombre, shape in shape_despues_poda.items():
    print(f"  {nombre}: {shape}")

print("\nTrain para XGBoost: real, sin SMOTE (ver 02b_ablacion_smote.ipynb):")
print(f"  X_train: {X_train.shape} | ataques: {prop_ataques_real:.4%} ({n_ataques_real} de {n_train_real})")

print("\nVersión con SMOTE, solo para la ablación de 02b (sampling_strategy=0.4):")
print(f"  X_train_smote: {X_train_smote.shape} | ataques: {prop_ataques_smote:.4%} "
      f"({n_ataques_smote} de {len(y_train_smote)})")
print(f"  Ejemplos sintéticos generados: {n_sinteticos}")

print("\nX_val y X_test: reales, sin cambios, listos para evaluar en Fase 6")
print(f"  X_val:  {X_val.shape}")
print(f"  X_test: {X_test.shape}")

RESUMEN — Fase 5: Feature Engineering (XGBoost)

Shapes antes de podar columnas:
  X_train: (310529, 61)
  X_val: (66541, 61)
  X_test: (66544, 61)

Shapes después de podar columnas (-6 columnas: Timestamp, bloque, Flow IAT Std, Bwd IAT Std, Bwd IAT Mean, Packet Length Std):
  X_train: (310529, 55)
  X_val: (66541, 55)
  X_test: (66544, 55)

Train para XGBoost: real, sin SMOTE (ver 02b_ablacion_smote.ipynb):
  X_train: (310529, 55) | ataques: 2.6503% (8230 de 310529)

Versión con SMOTE, solo para la ablación de 02b (sampling_strategy=0.4):
  X_train_smote: (423218, 55) | ataques: 28.5713% (120919 de 423218)
  Ejemplos sintéticos generados: 112689

X_val y X_test: reales, sin cambios, listos para evaluar en Fase 6
  X_val:  (66541, 55)
  X_test: (66544, 55)


### Guardado de los datos procesados

Se guardan `X_train`/`X_val`/`X_test` y `y_train`/`y_val`/`y_test` en `data/processed/xgboost/`, ya con las columnas podadas. `X_train.csv` / `y_train.csv` son el train **real, sin SMOTE**, que es el que usan los notebooks de entrenamiento de XGBoost. La versión con SMOTE se guarda aparte como `X_train_smote.csv` / `y_train_smote.csv` y solo la lee `02b_ablacion_smote.ipynb`. Así los notebooks de Fase 6 pueden leer directo estos archivos, sin repetir carga + split + poda + SMOTE cada vez que se quiera entrenar o probar algo.

(No quedan versionados en git, `data/processed/` ya está en `.gitignore`, así que cada quien los regenera corriendo este notebook en su propia máquina)

In [8]:
import os
import pandas as pd

output_dir = "../../data/processed/xgboost"
os.makedirs(output_dir, exist_ok=True)

X_datasets = {"X_train": X_train, "X_val": X_val, "X_test": X_test, "X_train_smote": X_train_smote}
y_datasets = {"y_train": y_train, "y_val": y_val, "y_test": y_test, "y_train_smote": y_train_smote}

rutas = {}
for nombre, X in X_datasets.items():
    ruta = os.path.join(output_dir, f"{nombre}.csv")
    X.to_csv(ruta, index=False)
    rutas[nombre] = ruta

for nombre, y in y_datasets.items():
    ruta = os.path.join(output_dir, f"{nombre}.csv")
    y.rename("label_binaria").to_csv(ruta, index=False, header=True)
    rutas[nombre] = ruta

print("Verificación (releer cada CSV y comparar shape con lo que está en memoria):\n")

for nombre, X in X_datasets.items():
    ruta = rutas[nombre]
    releido = pd.read_csv(ruta)
    assert releido.shape == X.shape, (
        f"{nombre}: shape guardado {releido.shape} != shape en memoria {X.shape}"
    )
    tamano_mb = os.path.getsize(ruta) / (1024 ** 2)
    print(f"  {nombre}: shape {releido.shape} OK | {ruta} ({tamano_mb:.2f} MB)")

for nombre, y in y_datasets.items():
    ruta = rutas[nombre]
    releido = pd.read_csv(ruta)["label_binaria"]
    assert releido.shape == y.shape, (
        f"{nombre}: shape guardado {releido.shape} != shape en memoria {y.shape}"
    )
    tamano_mb = os.path.getsize(ruta) / (1024 ** 2)
    print(f"  {nombre}: shape {releido.shape} OK | {ruta} ({tamano_mb:.2f} MB)")

Verificación (releer cada CSV y comparar shape con lo que está en memoria):



  X_train: shape (310529, 55) OK | ../../data/processed/xgboost\X_train.csv (80.52 MB)


  X_val: shape (66541, 55) OK | ../../data/processed/xgboost\X_val.csv (17.15 MB)


  X_test: shape (66544, 55) OK | ../../data/processed/xgboost\X_test.csv (17.14 MB)


  X_train_smote: shape (423218, 55) OK | ../../data/processed/xgboost\X_train_smote.csv (123.55 MB)
  y_train: shape (310529,) OK | ../../data/processed/xgboost\y_train.csv (0.89 MB)
  y_val: shape (66541,) OK | ../../data/processed/xgboost\y_val.csv (0.19 MB)
  y_test: shape (66544,) OK | ../../data/processed/xgboost\y_test.csv (0.19 MB)
  y_train_smote: shape (423218,) OK | ../../data/processed/xgboost\y_train_smote.csv (1.21 MB)
